# 07 · Integrate

This notebook puts the pieces together and draws the figure. It answers three questions:

1. Which transcripts carry each exon, and where does QKI bind next to it? (panels b and c, with notebook 02's windows)
2. How does QKI knockdown change each exon? (panel d, from notebook 06)
3. Do the two PICALM exons follow QKI's rule across the genome? (panel e, splice maps)

On Google Colab, run the next cell first in every session. It copies this repository onto the Colab machine and, with `SAVE_TO_DRIVE = True`, saves results and the figure to `My Drive/CS3_workflow` (Colab asks for permission to connect your Drive). `LITE = True` uses a small subset of the data that fits free Colab. On your own computer the cell does nothing.

In [ ]:
import sys

if "google.colab" in sys.modules:
    import os
    import subprocess
    REPO = "https://github.com/yeolab/encode-rbp-case-study"
    CLONE = "/content/encode-rbp-case-study"
    LITE = True    # True: small subset of the data, fits free Colab (about 1 h). False: all the data (5-8 h, ~50 GB)
    SAVE_TO_DRIVE = True    # results and the figure in My Drive/CS3_workflow; False: on this Colab machine only
    if not os.path.isdir(CLONE):
        subprocess.run(["git", "clone", "-q", "--depth", "1", REPO, CLONE], check=True)
    WORKFLOW = CLONE + "/notebooks/workflow"
    OUTPUT = None
    if SAVE_TO_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive")
        OUTPUT = "/content/drive/MyDrive/CS3_workflow"
    sys.path.insert(0, WORKFLOW)
    import colab_setup
    colab_setup.setup(WORKFLOW, lite=LITE, output_folder=OUTPUT)

In [ ]:
import json
import sys
from pathlib import Path

import pandas as pd
from IPython.display import Image, Markdown, display

sys.path.insert(0, ".")
import workflow_utils as U

ROOT, DATA = U.folders()
RESULTS = ROOT / "results" / "workflow"
API_CACHE = DATA / "downloads" / "rbpark_api"
print("case study folder:", ROOT)
print("data folder:", DATA)
import numpy as np

GENOME_CACHE = DATA / "downloads" / "genome"
DIRS = U.data_dirs(DATA)
EVENT_DIR, MAP_DIR, SIGNAL_DIR = DIRS["events"], DIRS["maps"], DIRS["signal"]
FIGURE = ROOT / "figures" / "workflow" / "CS3_PICALM_QKI_workflow"
import os

## 1. Isoforms and binding

GENCODE v40 lists PICALM's transcripts. Among the protein-coding ones, how many include or skip each exon?

In [ ]:
transcripts = U.gencode_transcripts(U.GENE, GENOME_CACHE)
isoforms = U.isoform_table(transcripts)
coding = isoforms[isoforms["type"] == "protein_coding"]
coding.drop(columns="type")

In [ ]:
coding[[c for c in coding.columns if c.endswith(("included", "skipped"))]].sum().rename("coding transcripts")

The MANE transcript (PICALM-202) includes the 150-nt exon and skips the 24-nt exon; no GENCODE transcript has both, though brain cDNA clones with both exist (Parikh et al. 2014).

QKI binds the sequence ACUAA, best as ACUAAY with a second UAAY a few nucleotides away (Galarneau & Richard 2005). The next cell counts these motifs in the 100 nt of intron next to each exon, on the RNA strand.

In [ ]:
import re

chrom, strand = U.LOCUS[:2]
e13, e24 = U.EXONS["exon13"]["grch38"][2:], U.EXONS["exon24nt"]["grch38"][2:]
regions = {"before the 150-nt exon": (e13[1], e13[1] + 100), "after the 24-nt exon": (e24[0] - 100, e24[0])}
for label, (start, end) in regions.items():
    rna = U.rna_sequence(chrom, start, end, strand, GENOME_CACHE)
    print(f"{label:24s} ACUAAY: {len(re.findall('ACUAA[CU]', rna))}  UAAY: {len(re.findall('UAA[CU]', rna))}")
    print("   ", rna)

## 2. Splice maps

A splice map averages eCLIP signal (IP minus input) across many exons, lined up on their splice sites. If QKI represses exons by binding before them, the exons included more after knockdown should have extra signal just before their 3′ splice site. If QKI promotes exons by binding after them, the exons excluded more should have extra signal just after their 5′ splice site. The gray band shows where 99% of random sets of background exons fall.

Two maps use the HepG2 knockdown, the stronger one: one with the ENCORE2 eCLIP (made in K562) and one with the ENCODE3 eCLIP from HepG2 itself. Each replicate takes about 10 minutes with rbp-maps' permutation test. Finished maps are read back from disk, so this cell needs the BAMs from notebook 03 only for maps that are missing.

In [ ]:
REPS = os.environ.get("REPS", "rep1,rep2").split(",")     # set REPS=rep1 to compute replicate 1 only
maps, tracks = {}, {}
for subject, kd_key in U.MAP_PAIRS:
    lists = U.event_lists(kd_key, EVENT_DIR)
    for rep in REPS:
        if U.map_is_finished(MAP_DIR, subject, kd_key, rep, lists):
            print(subject, kd_key, rep, "-> finished earlier, reading it back")
        else:
            print(subject, kd_key, rep, "-> computing (about 10 min)", flush=True)
            bams = pd.read_csv(RESULTS / "03_eclip_bams.tsv", sep="\t").set_index(["subject", "sample"])
            ip_sample, input_sample = U.ECLIP[subject]["reps"][rep]
            for sample in (ip_sample, input_sample):
                if (subject, sample) not in tracks:
                    row = bams.loc[(subject, sample)]
                    library = row[["mapped_F260", "mapped_records"]].to_dict() if U.LITE else None
                    tracks[(subject, sample)] = U.signal_tracks(row["bam"], lists[U.SETS[2]], SIGNAL_DIR, library=library)
            U.run_splice_map(tracks[(subject, ip_sample)], tracks[(subject, input_sample)], lists,
                             U.map_png(MAP_DIR, subject, kd_key, rep))
        maps[(subject, rep)] = U.load_splice_map(MAP_DIR, subject, kd_key, rep, lists)
display(Image(str(U.map_png(MAP_DIR, U.MAP_PAIRS[0][0], U.MAP_PAIRS[0][1], "rep1")), width=800))

The plot above is rbp-maps' own. Below, each PICALM exon's own row of the same data: where its eCLIP signal peaks, and how many positions of the genome-wide lines lie above the gray band.

In [ ]:
REGION = {"exon13": "3ss", "exon24nt": "5ss"}     # QKI binds before the 150-nt exon and after the 24-nt exon
genome_wide, picalm = [], []
for (subject, rep), m in maps.items():
    kd_key = dict(U.MAP_PAIRS)[subject]
    genome_wide.append({"eCLIP": subject, "rep": rep,
                        "included exons: positions above band, 100 nt before 3'ss": U.positions_above_band(
                            m["means"][U.SETS[0]], m["band"][1], "3ss"),
                        "excluded exons: positions above band, 100 nt after 5'ss": U.positions_above_band(
                            m["means"][U.SETS[1]], m["band"][1], "5ss")})
    m["traces"] = {}
    for exon, info in U.EXONS.items():
        trace, set_name = U.exon_trace(m["matrices"], exon)
        m["traces"][exon] = trace
        np.savetxt(MAP_DIR / f"{kd_key}_KD" / f"PICALM_{exon}_{subject}_{rep}.txt", trace)
        picalm.append({"eCLIP": subject, "rep": rep, "exon": info["name"], "group": set_name,
                       "peak (nt)": U.peak(trace, REGION[exon]), "max (x1e-4)": round(float(np.nanmax(trace)) * 1e4, 1)})
genome_wide, picalm = pd.DataFrame(genome_wide), pd.DataFrame(picalm)
genome_wide.to_csv(RESULTS / "07_splice_maps_genome_wide.tsv", sep="\t", index=False)
picalm.to_csv(RESULTS / "07_splice_maps_picalm.tsv", sep="\t", index=False)
display(genome_wide)
picalm

Expected for replicate 1. With the ENCORE2 eCLIP, the included exons are above the band at 85 of the 100 positions before the 3′ splice site and the excluded exons at 86 of 100 after the 5′ splice site; with the HepG2 eCLIP, 54 and 86. PICALM's own signal peaks 33 nt (ENCORE2) or 11 nt (HepG2) before the 150-nt exon, and 60 or 68 nt after the 24-nt exon. Both exons sit where the genome-wide rule says QKI binds.

## 3. The figure

`draw_figure()` (in `figure.py`) takes the windows from notebook 02, the transcripts above, the numbers from notebook 06 and the replicate-1 maps.

In [ ]:
from figure import draw_figure

windows = pd.read_csv(RESULTS / "02_qki_windows_PICALM.tsv", sep="\t")
locus = {"exons": {exon: list(info["grch38"][2:]) for exon, info in U.EXONS.items()}, "transcripts": transcripts,
         "windows": windows[["subject", "start", "end", "log2_enrichment", "q_value"]].values.tolist()}
knockdown = json.loads((RESULTS / "06_picalm_knockdown.json").read_text())
labels = {"QKI_K562_5007": "QKI ENCORE2\n(K562) eCLIP ×\nHepG2 KD", "QKI_HepG2_ENCSR570WLM": "QKI ENCODE3\n(HepG2) eCLIP ×\nHepG2 KD"}
rows_e = [{"label": labels[subject], "means": maps[(subject, "rep1")]["means"], "band": maps[(subject, "rep1")]["band"],
           "traces": maps[(subject, "rep1")]["traces"]} for subject, _ in U.MAP_PAIRS]
png = draw_figure(locus, knockdown, rows_e, FIGURE)
print("saved", png, "and .pdf")
display(Image(png, width=900))